### QuSciTech-Labs — Navigation

[Public Labs](https://github.com/jopaneur/quscitech-labs) ·
[Full Edition Access](https://github.com/jopaneur/quscitech-labs#-full-edition-kdp) ·
[Private Repo](https://github.com/jopaneur/quscitech-labs-full) ·
[QuSciTech.com](https://www.quscitech.com) ·
[The Quantum AI Book (QAIS)](https://www.amazon.com/dp/placeholder) ·

DOI: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.17212825.svg)](https://doi.org/10.5281/zenodo.17212825)

### E.2 Lab 4 — Fidelity & Trace Distance — Similarity vs Perturbation

### Lab Access and Execution Guide
This guide explains how to run and explore the hands-on quantum computing labs that accompany the book  
**Quantum AI Systems: Theory, Architecture, and Applications** (Professional Volume).  

The labs are an integral part of the MyQuantumBook project, designed to reinforce key concepts from the chapters through interactive exploration. They are built for execution on **Google Colab** and **IBM Quantum backends** using **Qiskit**, and follow the IEEE-compliant figure, caption, and documentation standards described in the text.  

Each lab is cross-referenced to its corresponding chapter and appendix figure (Appendix E), ensuring reproducibility and scholarly traceability.  

**Getting Started**
1. Launch the notebook in Google Colab using the provided badge.
2. Run the setup cells to install Qiskit:
   `!pip install qiskit`

**Using IBM Quantum Systems**
1. Sign up at https://quantum.ibm.com and create an API token.
2. Run the IBMQ setup cell.
3. Replace 'MY_API_TOKEN' with your real token (only needed once).
4. Select backends using `provider.get_backend('ibmq_qasm_simulator')` or others.

**Lab Structure**
Each code section aligns with a chapter from the book.
- Modify and re-run code blocks.
- View circuits with `.draw()`.
- Apply to custom inputs to deepen your understanding.

**Additional Help**
- Refer to the Qiskit Documentation: https://qiskit.org/documentation/
- For support, contact your course instructor or visit the IBM Quantum Community forums.

3. Or launch this lab directly now: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jopaneur/QuantumAI-Labs/blob/main/Advanced_Labs/notebooks/Chapter_2_Operations_&_Scientific_Framework_of_QAIS_Advanced_Challenge_Bloch_Trajectories_Under_Composite_Gates.ipynb)



---
**Note for Lab Participants**
Each plot generated in this notebook is automatically saved as a `.png` file under: Advanced_Labs/figures/

The filenames follow the Appendix E figure numbering (e.g., `E2_1_Bloch_Trajectories.png`, `E2_6_DensityMatrix_Heatmap.png`).  

This allows you to both view results inline in Colab **and** find the corresponding image files for reports, submissions, or cross-references in the book.

**Where Figures Are Saved**
- In **Google Colab**, the images are created inside the session’s working directory at:  
  `/content/Advanced_Labs/figures/`  

- When running **locally**, they appear next to your notebook files, under the subfolder:  
  `Advanced_Labs/figures/`  

- These images are **not automatically added to your GitHub repo**. They will only appear there if you manually copy, commit, and push them.

**Customizing Save Location**
If you want the figures saved elsewhere, you can change the `subdir` default in the `save_e_figure()` helper or pass a different path each time you call it.


---

**Book Reference: Chapter 9 — Quantum Encoding & Information Metrics**

*Chapter 9* introduces information metrics as the analytical foundation of QAIS, providing quantitative tools to evaluate how closely two quantum states resemble each other. It defines fidelity and trace distance as complementary measures of similarity and divergence—key diagnostics for assessing the accuracy, stability, and integrity of encoded quantum information.
Fidelity expresses how well a prepared or transmitted state matches a target, while trace distance measures the statistical distinguishability between states. Together, they form a dual perspective: one geometric (overlap in Hilbert space) and one probabilistic (difference in measurement outcomes).

This chapter emphasizes that QAIS performance must be judged not only by computational output but also by information preservation. Small perturbations in amplitude or phase can alter outcomes dramatically, and these metrics make such effects visible.


**Beginner Lab 9 — Fidelity & Trace Distance: Similarity vs Perturbation**

Beginner Lab 9 applies these ideas experimentally. Learners encode quantum states, introduce controlled perturbations, and compute both fidelity and trace distance to quantify the resulting divergence. The exercise links measurement theory to practical QAIS benchmarking, showing how resilience and accuracy can be expressed numerically rather than qualitatively.

*Goal:* Compute and compare fidelity and trace distance between pairs of quantum states under varying levels of perturbation. Learners prepare an initial reference state, apply controlled rotations or noise to generate perturbed versions, and then calculate both metrics.

**Expected Outcome:**

* Fidelity decreases as perturbations grow, indicating loss of similarity.

* Trace distance increases proportionally, reflecting greater distinguishability.

* The complementary trends confirm that these two metrics provide a complete view of state divergence.

This lab reinforces Chapter 9’s theme that information quality in QAIS is measurable and quantifiable. By connecting geometry (fidelity) and probability (trace distance), learners develop the intuition that maintaining high overlap and low distance defines the operational stability of quantum intelligence. Cross-reference: Appendix E.1, Figure E.1.9 — Fidelity and Trace Distance under Perturbation.

---

In [ ]:
# ---- Figure helper (robust; use in every coded lab) ----
import os, matplotlib.pyplot as plt

def save_e_figure(fig_label: str,
                  fname: str,
                  subdir: str = "Beginner_Labs/figures",
                  fig=None, ax=None):
    """Save the current/explicit figure with a prefixed label and consistent path."""
    os.makedirs(subdir, exist_ok=True)
    if fig is None:
        fig = plt.gcf()
    if ax is None:
        ax = fig.axes[0] if fig.axes else None
    if ax is None:
        print("⚠️ No axes found. Draw a plot first, or pass fig/ax explicitly.")
        return
    title = ax.get_title() or ""
    if not title.startswith(fig_label):
        ax.set_title((fig_label + " — " + title).strip(" —"))
    outpath = os.path.join(subdir, fname)
    fig.tight_layout()
    fig.savefig(outpath, dpi=160)
    print("Saved", outpath)


In [ ]:
# === Environment Setup ===
import sys, subprocess, pkgutil
def ensure(pkg):
    if pkg not in {m.name for m in pkgutil.iter_modules()}:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p in ['qiskit','qiskit-aer','matplotlib','numpy','scikit-learn']:
    ensure(p)
import qiskit, numpy as np, matplotlib.pyplot as plt
print('Python:', sys.version.split()[0])
print('Qiskit:', qiskit.__version__)


---


**Lab 9: Fidelity & Trace Distance**

Technical Spotlight — “Measuring Closeness of Quantum States”
This lab introduces fidelity (how much one state resembles another) and trace distance (how far they are apart). The plot shows fidelity staying near 1 when perturbations are small, while trace distance gradually increases as states diverge. The process flow: prepare an entangled state, slightly perturb it by rotating qubits, then compute both metrics. The contrasting curves highlight two complementary perspectives on state comparison: fidelity emphasizes overlap, while trace distance emphasizes distinguishability. This visualization is essential because it demonstrates robustness — a small deviation barely affects fidelity but still registers as measurable change in trace distance.

*Book Reference: Chapter 9 — Quantum Encoding and Information Metrics for AI*

This lab visualizes how fidelity and trace distance quantify similarity between states. It gives practical intuition for Chapter 9’s information-theoretic metrics and their role in benchmarking QAIS models.

**Expected Results**

At φ = 0, the perturbed state equals the reference, so F = 1 and T = 0.

As φ increases, Fidelity decreases smoothly; Trace distance increases smoothly.

Analytically for this circuit, F = cos²(φ/2) and T = |sin(φ/2)|, so the two curves are complementary and monotonic on 0 ≤ φ ≤ π.

In [ ]:

# Lab 9 — Fidelity and Trace Distance

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np
import matplotlib.pyplot as plt

# --- Reference (baseline) Bell state |Φ+> = (|00> + |11>)/√2 ---
def prepare_reference() -> np.ndarray:
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    return Statevector.from_instruction(qc).data

# --- Perturbed state: insert a phase Rz(φ) before entangling ---
def prepare_perturbed(phi: float) -> np.ndarray:
    q = QuantumCircuit(2)
    q.h(0)
    q.rz(phi, 0)
    q.cx(0, 1)
    return Statevector.from_instruction(q).data

# --- Sweep φ and compute Fidelity F and Trace distance T ---
psi = prepare_reference()
phis = np.linspace(0.0, 0.6, 13)  # radians (0 to 0.6 ~ 34.4°), adjust as desired

def fidelity(p: np.ndarray, q: np.ndarray) -> float:
    # For pure states: F = |<p|q>|^2
    return float(abs(np.vdot(p, q))**2)

def trace_distance_from_F(F: float) -> float:
    # For pure states: T(ρ,σ) = sqrt(1 - F)
    return float(np.sqrt(max(0.0, 1.0 - F)))

fids = []
tds  = []
for phi in phis:
    phi_state = prepare_perturbed(phi)
    F = fidelity(psi, phi_state)
    T = trace_distance_from_F(F)
    fids.append(F)
    tds.append(T)

# Optional quick check
print(list(zip(np.round(phis, 3), np.round(fids, 3), np.round(tds, 3))))

# --- Plot ---
plt.plot(phis, fids, marker="o", label="Fidelity  F(ψ,ψφ)")
plt.plot(phis, tds, marker="s", label="Trace distance  T(ψ,ψφ)")
plt.xlabel("Phase perturbation φ (radians)")
plt.ylabel("Value")
plt.title("Fidelity and Trace Distance vs Phase Perturbation")

plt.legend()

# --- Save with correct IEEE label ---
fig, ax = plt.gcf(), plt.gca()
save_e_figure("Figure E.1.9", "P1_Lab09_Fidelity_TraceDistance.png",
              subdir="Beginner_Labs/figures", fig=fig, ax=ax)

plt.show()


Figure E.1.9 — Fidelity and Trace Distance vs Phase Perturbation.
The curves show how a relative phase rotation φ applied before entanglement changes the state’s similarity to the reference Bell state. Fidelity starts at 1 when φ = 0 and decreases as φ grows, while the trace distance starts at 0 and increases. For pure states these quantities are linked by T = √(1 − F), producing complementary trends.

**Methodology Analysis**

Prepare a reference Bell state ∣Φ⁺⟩ via H on qubit 0 followed by CX(0→1). Create perturbed states by inserting Rz(φ) on qubit 0 before the CNOT, then generate the statevector. For a sweep of φ values, compute:

Fidelity F(ψ, ψφ) = |⟨ψ ∣ ψφ⟩|²

Trace distance T(ψ, ψφ) = √(1 − F) (valid for pure states)
Plot F and T versus φ to visualize sensitivity to coherent phase errors.

**Technical Analysis (for the Visual)**

The circuit with Rz(φ) before CX produces the state
∣ψφ⟩ = (∣00⟩ + e^{iφ}∣11⟩)/√2,
so the overlap with the reference ∣Φ⁺⟩ is
⟨Φ⁺ ∣ ψφ⟩ = (1 + e^{iφ})/2, giving F = |(1 + e^{iφ})/2|² = (1 + cos φ)/2 = cos²(φ/2).
For pure states, the trace distance between density matrices equals T = √(1 − F) = |sin(φ/2)|.
Interpretation: a small coherent phase error on a single qubit before entanglement rotates probability amplitude between the ∣00⟩ and ∣11⟩ components, smoothly reducing similarity to the target and increasing distinguishability.

**Intuition Sidebar**

Picture a spotlight aimed straight at a mirror; the reflection is brightest when perfectly aligned. Twisting the mirror slightly dims the reflection. Here, φ is the “twist”: when φ = 0 the states match perfectly (F = 1, T = 0). As φ grows, the match fades and the distance grows, reflecting how a simple phase drift makes two states more distinguishable.

**Conclusion — Lab 9**

This lab shows how coherent phase errors translate into quantifiable changes in state similarity. By plotting Fidelity and Trace distance against the phase offset, learners see complementary measures of closeness that are analytically related for pure states. In practice, these metrics guide calibration, model validation, and robustness checks for quantum AI pipelines where small phase deviations can impair performance.

**Key Takeaways**

For pure states, Fidelity and Trace distance are linked: T = √(1 − F).

A phase rotation before entanglement smoothly reduces similarity to the target state: F ↓, T ↑ with φ.

These metrics are practical diagnostics for error sensitivity and model robustness in quantum AI systems.

**Congratulations — Lab 9**

Congratulations on completing Lab 9 — Fidelity and Trace Distance! By working through this exercise, you have not only explored two of the most important metrics in quantum information but also built the intuition to see how small phase shifts directly affect state similarity and distinguishability. This mastery of fidelity and trace distance equips you with analytical tools that are widely applied in quantum error analysis, quantum benchmarking, and AI model validation. Keep building on this foundation—the skills you are developing here are those used by researchers and professionals pushing the frontiers of quantum AI systems.

### Appendix E → Appendix B Cross-Reference
See **Appendix B — Quick Self-Check, Chapter 9 — Quantum Encoding & Information Metrics**:  
- Review Questions 1 and 2 (fidelity definition and trace-distance behavior).  
They correspond to the divergence-quantification experiment in **E.2 Lab 4**, where fidelity decreases and trace distance increases under perturbation (Figure E.2.4).


---
**How to save or submit your work**

- **If you are a student (graded/evaluated):**  
  1. Export your key plots or the entire notebook to PDF (File → Print/Save as PDF).  
  2. Save the notebook (`.ipynb`).  
  3. Bundle any extra files (CSVs/images) if used.  
  4. Upload to your LMS or repository as instructed (include your name and lab number).  
  5. Repro checklist: set a random seed where applicable, note backend and shots, and list package versions.  

- **If you are a professional/self‑learner (non‑graded exercise):**  
  1. Save the notebook (`File → Download .ipynb`) to your computer for personal reference.  
  2. Optionally export to PDF for archiving.  
  3. Keep any generated plots or data locally.  
  4. Use version control (GitHub, GitLab) if you wish to track your personal progress.


---
